AI agent for giving recipes for leftoever foods. from fridge, with memory 

In [8]:
#initial model set up
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ.get("GROQ_API_KEY")
model   = ChatGroq(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            
    )


python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7


In [9]:
#intialozinf human message 
from langchain.messages import HumanMessage

response = model.invoke([HumanMessage(content="What is the capital of France?")])
print(response.content)


The capital of France is Paris.


In [10]:
#creating tools before creating agents
from langchain.tools import tool
from tavily import TavilyClient
from typing import Any
tavelyclient = TavilyClient(api_key=os.environ.get("TAVELY_API_KEY"))
@tool
def websearchtool(query:str)->str:
    """this tool performs a web search and returns the result"""
    
    return tavelyclient.search(query)

In [11]:
#initiliased memory and agent
from langgraph.checkpoint.memory import InMemorySaver

from langchain.agents import create_agent

agent = create_agent(model = model,
                     system_prompt = "You are a helpful assistant.",
                     checkpointer = InMemorySaver(),
                     tools = [websearchtool]
                     
                     )

In [14]:
#code to get image input from user
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.jpg', multiple=False)
display(uploader)

FileUpload(value=(), accept='.jpg', description='Upload')

In [15]:
#converting the image into base64 string to pass it as an input to the agent
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [16]:
#giving multimodel question, the agent will use the websearch tool to find the best recipe for the leftover food in the image and also answer any follow up question related to it
#this is the template of the multimodel question

multimodal_question = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "from all the leftover food in the image[fridge], do a websearch and find the best recipe for it, and can give me answers for any follow  up quesion related to it"
        },
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{img_b64}"
            }
        }
    ]
)

In [17]:
#invoke the agent with the multimodal question
response = agent.invoke(
    {
        "messages": [multimodal_question]
    },
    config={
        "configurable": {
            "thread_id": "1"
        }
    }
)

print(response["messages"][-1].content)

Here are some recipe ideas you can make using the ingredients in your fridge:

1. **Tomato, Orange and Cucumber Salad**: A fresh salad of grape tomatoes, cucumber, and navel orange is not only full of flavor and nutrition, it's a colorful addition to any dinner table.
2. **Green salad with oranges and grapes**: A sweet and refreshing salad that combines oranges, grapes, and greens.
3. **Grape & Orange Salad**: A simple salad that combines oranges, grapes, and fennel, with a citrus vinaigrette.
4. **Mixed Green Salad with Oranges & Grapes**: A bright, colorful salad of mixed greens, red onion, oranges and grapes tossed in a Spanish-inspired vinaigrette and garnished with herbs and blue cheese.
5. **Grape and Tomato Salad**: A fresh and vibrant salad that brings together sweet and juicy grapes, tomatoes, and herbs.

Feel free to ask me any follow-up questions about these recipes, such as "What type of dressing should I use for the Grape & Orange Salad?" or "Can I substitute the oranges w

In [21]:
#giving another query
message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "can you tell me the recipe[Tomato, Orange and Cucumber Salad] process more clearly"
        },
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{img_b64}"
            }
        }
    ]
)

In [ ]:
secondresponse = agent.invoke(
    {"messages": [message]},
    config={"configurable": {"thread_id": "1"}}
)


ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id